# Pipeline de descarga — Wikiroutes Lima

Corre las celdas en orden para una actualización completa.
Cada celda es independiente: se puede correr sola si solo necesitas regenerar ese paso.

| Celda | Script | Produce |
|-------|--------|---------|
| 1 | `wr_build_catalog.py` | carpetas `route_*` + `wr_map.json` + `wr_overrides.json` |
| 2 | `wr_sync_indexes.py` | `wr_map.json` + `wr_overrides.json` (regeneración completa) |
| 3 | `wr_build_codes.py` | `wr_codes_master.csv` |
| 4 | `wr_build_extremes.py` | `wr_extremes.json` |
| 5 | verificación | resumen del estado actual |

> **Nota sobre las celdas 1 y 2**: la celda 1 actualiza `wr_map.json` incrementalmente
> solo para las rutas nuevas. La celda 2 regenera `wr_map.json` completo desde cero
> a partir de todas las carpetas en disco. Correr solo la celda 2 es suficiente
> si ya tienes las carpetas descargadas y solo quieres regenerar los índices.

In [ ]:
# Celda 1: Descarga de rutas nuevas desde Wikiroutes
# Abre Chrome visible, recorre el catálogo, y descarga solo las rutas que no existen.
# Cambia MAX_NUEVAS para limitar la cantidad en corridas de prueba.

# MAX_NUEVAS = 10  # descomentar para prueba
!python wr_build_catalog.py

In [ ]:
# Celda 2: Regenera wr_map.json y wr_overrides.json desde todas las carpetas en disco
# Usar cuando se cambia la lógica de display_id o color, o después de correcciones manuales.

!python wr_sync_indexes.py

In [ ]:
# Celda 3: Genera wr_codes_master.csv con el matching Wikiroutes <-> ATU

!python wr_build_codes.py

In [ ]:
# Celda 4: Genera wr_extremes.json con los pares Origen -> Destino de cada ruta

!python wr_build_extremes.py

In [ ]:
# Celda 5: Verificación del estado actual
import json
from pathlib import Path

def find_root():
    cur = Path.cwd()
    for _ in range(8):
        if (cur / "data" / "processed" / "transporte").is_dir():
            return cur
        cur = cur.parent
    raise RuntimeError("No se encontró la raíz del repo")

ROOT = find_root()

carpetas = list((ROOT / "data" / "processed" / "transporte").glob("route_*"))
print(f"Carpetas route_* en disco:    {len(carpetas)}")

wr_map_path = ROOT / "pipeline" / "output" / "wr_map.json"
if wr_map_path.exists():
    wr_map = json.loads(wr_map_path.read_text(encoding="utf-8"))
    print(f"Entradas en wr_map.json:      {len(wr_map.get('routes', {}))}")
else:
    print("wr_map.json: no encontrado")

extremes_path = ROOT / "pipeline" / "output" / "wr_extremes.json"
if extremes_path.exists():
    extremes = json.loads(extremes_path.read_text(encoding="utf-8"))
    print(f"Entradas en wr_extremes.json: {len(extremes)}")
else:
    print("wr_extremes.json: no encontrado")

csv_path = ROOT / "pipeline" / "output" / "wr_codes_master.csv"
if csv_path.exists():
    lineas = csv_path.read_text(encoding="utf-8").splitlines()
    print(f"Filas en wr_codes_master.csv: {len(lineas) - 1} (sin cabecera)")
else:
    print("wr_codes_master.csv: no encontrado")